# Motion-Blindness Benchmark — Video-LLaVA Evaluation
**Owais Ahmad | Islamia University Bahawalpur | 2026**

Evaluates Video-LLaVA-7B on the Motion-Blindness benchmark across all 4 question categories.

### Before running
- Settings → Internet → **ON**
- Settings → Accelerator → **GPU T4 x1**
- Upload your `rgb_frames/` folder and `annotation_spreadsheet.csv` as a Kaggle dataset
- Update `DATASET_PATH` in Cell 2

### Expected runtime: 4–6 hours for 70 clips on T4
Checkpoints saved every 10 clips — safe to resume if session times out.

## Cell 1 — Install Dependencies

In [ ]:
%%capture
# transformers 4.52.0+ required for Kaggle numpy 2.4.6 compatibility
!pip install transformers==4.52.0 accelerate==0.30.0 av==12.0.0 -q

import transformers
print(f'transformers: {transformers.__version__}')
print('Install complete')

## Cell 2 — Configuration
⚠️ Update `DATASET_PATH` before running anything else.

In [ ]:
import os

# ── UPDATE THIS to your Kaggle dataset name ──────────────────────────────────
DATASET_PATH = '/kaggle/input/motion-blindness-frames'
# ─────────────────────────────────────────────────────────────────────────────

RGB_FRAMES_DIR      = os.path.join(DATASET_PATH, 'rgb_frames')
ANNOTATION_CSV_PATH = os.path.join(DATASET_PATH, 'annotation_spreadsheet.csv')
OUTPUT_DIR          = '/kaggle/working'
OUTPUT_CSV          = os.path.join(OUTPUT_DIR, 'video_llava_results.csv')
CHECKPOINT_CSV      = os.path.join(OUTPUT_DIR, 'video_llava_checkpoint.csv')
MODEL_ID            = 'LanguageBind/Video-LLaVA-7B-hf'
FRAMES_TO_SEND      = 8     # evenly-spaced frames per clip sent to model
SAVE_EVERY          = 10    # save checkpoint every N clips
MAX_NEW_TOKENS      = 5     # model only needs to output A/B/C/D

if not os.path.exists(DATASET_PATH):
    print(f'ERROR: Dataset not found: {DATASET_PATH}')
    print('Update DATASET_PATH to match your Kaggle dataset slug.')
    print('Format: /kaggle/input/YOUR-DATASET-SLUG/')
else:
    print(f'Dataset found: {DATASET_PATH}')
    print(f'Contents: {os.listdir(DATASET_PATH)}')

## Cell 3 — Verify GPU

In [ ]:
import torch

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:  {name}')
    print(f'VRAM: {vram:.1f} GB')
    if vram < 14:
        print('WARNING: Need T4 (16GB). Enable in Settings -> Accelerator.')
    else:
        print('OK: sufficient VRAM for Video-LLaVA-7B')
else:
    print('ERROR: No GPU found. Enable T4 in Settings -> Accelerator.')

## Cell 4 — Load Model
⚠️ First run downloads ~14GB. Takes 5–10 min. Subsequent runs use cache.

In [ ]:
from transformers import VideoLlavaProcessor, VideoLlavaForConditionalGeneration
import torch

print(f'Loading: {MODEL_ID}')

processor = VideoLlavaProcessor.from_pretrained(MODEL_ID)
print('Processor loaded')

model = VideoLlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # float16 fits in 16GB VRAM
    device_map='auto',           # auto-distribute layers on GPU
    low_cpu_mem_usage=True,
)
model.eval()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f'GPU memory used: {allocated:.1f} GB')

print('Model ready')

## Cell 5 — Helper Functions

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image


def get_evenly_spaced_frames(frame_dir, n):
    """Get n evenly-spaced frame paths from a clip folder."""
    frames = sorted([
        os.path.join(frame_dir, f)
        for f in os.listdir(frame_dir)
        if f.endswith('.jpg') or f.endswith('.png')
    ])
    if not frames:
        return []
    if len(frames) <= n:
        return frames
    indices = [int(i * (len(frames) - 1) / (n - 1)) for i in range(n)]
    return [frames[i] for i in indices]


def load_frames_as_numpy(frame_paths):
    """
    Load frames as uint8 numpy array shaped (T, H, W, 3).
    VideoLlavaProcessor expects this exact format.
    """
    frames = []
    for path in frame_paths:
        img = Image.open(path).convert('RGB').resize((224, 224))
        frames.append(np.array(img, dtype=np.uint8))
    return np.stack(frames, axis=0)   # (T, 224, 224, 3)


def build_prompt(question_text, options_text):
    """Build prompt in VideoLLaVA chat format."""
    return (
        'USER: <video>\n'
        f'{question_text}\n\n'
        f'Options:\n{options_text}\n\n'
        'Answer with only the letter A, B, C, or D.\n'
        'ASSISTANT:'
    )


def extract_answer(generated_text):
    """Extract A/B/C/D from model output."""
    text = generated_text.strip().upper()
    # Answer appears after ASSISTANT: tag
    if 'ASSISTANT:' in text:
        after = text.split('ASSISTANT:')[-1].strip()
        for ch in after:
            if ch in {'A', 'B', 'C', 'D'}:
                return ch
    # Fallback: scan full output
    for ch in text:
        if ch in {'A', 'B', 'C', 'D'}:
            return ch
    return 'INVALID'


def load_checkpoint(path):
    """Return set of clip_ids already evaluated."""
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    done = set(df['clip_id'].tolist())
    print(f'Checkpoint found: {len(done)} clips already done')
    return done


def save_checkpoint(results, path):
    if results:
        pd.DataFrame(results).to_csv(path, index=False)


print('Helper functions ready')

## Cell 6 — Load Annotation Spreadsheet

In [ ]:
annotation_df = pd.read_csv(ANNOTATION_CSV_PATH)

if 'include_in_benchmark' in annotation_df.columns:
    annotation_df = annotation_df[annotation_df['include_in_benchmark'] != 'no']

print(f'Clips loaded: {len(annotation_df)}')

motion_col = 'verified_depth_motion' if 'verified_depth_motion' in annotation_df.columns else 'depth_motion'
print(f'Depth motion distribution:')
print(annotation_df[motion_col].value_counts().to_string())

print(f'\nSample Cat 3 question:')
print(annotation_df.iloc[0]['q3_text'])
print(f'Correct: {annotation_df.iloc[0]["q3_correct"]}')

## Cell 7 — Single Clip Test
Run this first to confirm the model processes frames correctly before the full loop.

In [ ]:
test_row    = annotation_df.iloc[0].to_dict()
clip_id     = test_row['clip_id']
rgb_dir     = os.path.join(RGB_FRAMES_DIR, clip_id)

print(f'Testing: {clip_id}')
print(f'Q (Cat 3): {test_row.get("q3_text", "")}')
print(f'Correct:   {test_row.get("q3_correct", "")}')
print()

frame_paths = get_evenly_spaced_frames(rgb_dir, FRAMES_TO_SEND)
video_array = load_frames_as_numpy(frame_paths)
print(f'Video array shape: {video_array.shape}')  # expect (8, 224, 224, 3)

prompt = build_prompt(test_row.get('q3_text', ''), test_row.get('q3_options', ''))

inputs = processor(
    text=prompt,
    videos=video_array,
    return_tensors='pt',
    padding=True,
).to(model.device, torch.float16)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=1.0,
    )

generated = processor.decode(output_ids[0], skip_special_tokens=True)
answer    = extract_answer(generated)

print(f'Model answer: {answer}')
print(f'Correct:      {test_row.get("q3_correct", "")}')
print(f'Is correct:   {answer == test_row.get("q3_correct", "")}')
print(f'Raw tail: ...{generated[-80:]}')

torch.cuda.empty_cache()
print('\nTest passed — ready for full evaluation.')

## Cell 8 — Full Evaluation Loop
Evaluates every clip on all 4 questions. Checkpoints every 10 clips.
**If interrupted: just re-run this cell — resumes from checkpoint automatically.**

In [ ]:
from tqdm.notebook import tqdm

QUESTIONS = [
    ('q1', 'Cat 1: 2D Direction'),
    ('q2', 'Cat 2: 2D Speed'),
    ('q3', 'Cat 3: Depth Direction'),
    ('q4', 'Cat 4: Depth Rate'),
]

# Resume support
already_done = load_checkpoint(CHECKPOINT_CSV)
all_results  = []
if already_done and os.path.exists(CHECKPOINT_CSV):
    all_results = pd.read_csv(CHECKPOINT_CSV).to_dict('records')

remaining = annotation_df[~annotation_df['clip_id'].isin(already_done)]
print(f'To evaluate: {len(remaining)} clips | Already done: {len(already_done)}')
print(f'Estimated time: {len(remaining) * 4 * 6 / 3600:.1f} hours')
print()

for clip_idx, (_, row) in enumerate(tqdm(remaining.iterrows(),
                                          total=len(remaining),
                                          desc='Clips')):
    clip_id = row['clip_id']
    rgb_dir = os.path.join(RGB_FRAMES_DIR, clip_id)

    if not os.path.exists(rgb_dir):
        tqdm.write(f'  SKIP {clip_id}: dir not found')
        continue

    frame_paths = get_evenly_spaced_frames(rgb_dir, FRAMES_TO_SEND)
    if not frame_paths:
        tqdm.write(f'  SKIP {clip_id}: no frames')
        continue

    try:
        video_array = load_frames_as_numpy(frame_paths)
    except Exception as e:
        tqdm.write(f'  SKIP {clip_id}: {str(e)[:50]}')
        continue

    clip_result = {
        'clip_id':      clip_id,
        'model':        'Video-LLaVA-7B',
        'condition':    'A',
        'scene_type':   row.get('scene_type', ''),
        'depth_motion': row.get('verified_depth_motion', row.get('depth_motion', '')),
        'speed':        row.get('speed', ''),
    }

    for qkey, qlabel in QUESTIONS:
        qtext    = row.get(f'{qkey}_text', '')
        qopts    = row.get(f'{qkey}_options', '')
        qcorrect = row.get(f'{qkey}_correct', '')

        if not qtext:
            clip_result[f'{qkey}_answer']         = 'NO_QUESTION'
            clip_result[f'{qkey}_correct_answer'] = qcorrect
            clip_result[f'{qkey}_is_correct']     = False
            continue

        prompt = build_prompt(qtext, qopts)

        try:
            inputs = processor(
                text=prompt,
                videos=video_array,
                return_tensors='pt',
                padding=True,
            ).to(model.device, torch.float16)

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    temperature=1.0,
                )

            generated = processor.decode(output_ids[0], skip_special_tokens=True)
            answer    = extract_answer(generated)

        except torch.cuda.OutOfMemoryError:
            tqdm.write(f'  OOM: {clip_id} {qkey} — clearing cache')
            torch.cuda.empty_cache()
            answer = 'OOM_ERROR'
        except Exception as e:
            tqdm.write(f'  ERROR: {clip_id} {qkey}: {str(e)[:60]}')
            answer = 'ERROR'

        clip_result[f'{qkey}_answer']         = answer
        clip_result[f'{qkey}_correct_answer'] = qcorrect
        clip_result[f'{qkey}_is_correct']     = (answer == qcorrect)

    all_results.append(clip_result)

    if (clip_idx + 1) % SAVE_EVERY == 0:
        save_checkpoint(all_results, CHECKPOINT_CSV)
        tmp = pd.DataFrame(all_results)
        c3  = tmp['q3_is_correct'].mean() * 100 if 'q3_is_correct' in tmp.columns else 0
        c4  = tmp['q4_is_correct'].mean() * 100 if 'q4_is_correct' in tmp.columns else 0
        tqdm.write(f'  [Checkpoint {clip_idx+1}] Cat3: {c3:.1f}%  Cat4: {c4:.1f}%  (saved)')

    torch.cuda.empty_cache()

print(f'Evaluation complete. Total: {len(all_results)} clips')

## Cell 9 — Save Final Results and Print Summary

In [ ]:
import numpy as np

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}  ({len(results_df)} clips)')

print('\n=== Video-LLaVA Accuracy ===')
cats = {
    'q1': 'Cat 1: 2D Direction ',
    'q2': 'Cat 2: 2D Speed     ',
    'q3': 'Cat 3: Depth Dir    ',
    'q4': 'Cat 4: Depth Rate   ',
}
for q, label in cats.items():
    col = f'{q}_is_correct'
    if col in results_df.columns:
        acc = results_df[col].mean() * 100
        print(f'  {label}: {acc:.1f}%')

if all(f'{q}_is_correct' in results_df.columns for q in ['q1','q2','q3','q4']):
    avg_2d    = np.mean([results_df['q1_is_correct'].mean(),
                         results_df['q2_is_correct'].mean()]) * 100
    avg_depth = np.mean([results_df['q3_is_correct'].mean(),
                         results_df['q4_is_correct'].mean()]) * 100
    gap = avg_2d - avg_depth
    print(f'\n  2D avg:    {avg_2d:.1f}%')
    print(f'  Depth avg: {avg_depth:.1f}%')
    print(f'  Gap:      -{gap:.1f} pp')
    status = 'CONFIRMED' if avg_depth < 35 else 'CHECK — higher than expected'
    print(f'  Near-chance on depth-axis: {status}')

print('\n--- By depth motion type (Cat 3 only) ---')
mc = 'depth_motion'
if mc in results_df.columns and 'q3_is_correct' in results_df.columns:
    for mt in ['toward', 'away', 'stationary']:
        sub = results_df[results_df[mc] == mt]
        if len(sub):
            c3 = sub['q3_is_correct'].mean() * 100
            print(f'  {mt:<12}: {c3:.1f}%  (n={len(sub)})')

## Cell 10 — Download and Next Steps

### Download results
1. Click **Output** tab (right side of this page)
2. Download `video_llava_results.csv`
3. Copy to your local machine:
```
C:\Users\Owais\Documents\motion-blindness\results\raw_outputs\video_llava_results.csv
```
4. Run locally:
```bash
python scripts/stage6_compile_results.py
python scripts/stage7_visualize.py
```

### If session timed out
Re-run **Cell 8 only** — checkpoint auto-resumes.

### Expected results
| Category | Hypothesis |
|---|---|
| Cat 1: 2D Direction | 45–52% |
| Cat 2: 2D Speed | 35–42% |
| **Cat 3: Depth Direction** | **25–30% (near chance)** |
| **Cat 4: Depth Rate** | **24–28% (near chance)** |
| **2D → Depth gap** | **~20 percentage points** |